## Step 1 — Install libraries

In [ ]:
# Install required libraries
!pip -q install pypdf
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install transformers accelerate
!pip -q install rank_bm25
!pip -q install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 57.4 MB/s eta 0:00:00


Installs all the required Python libraries needed for the project.

## Step 2 — Imports

In [ ]:
import re
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rank_bm25 import BM25Okapi


Imports all necessary libraries for document processing, embeddings, vector search, and language model tasks.

## Step 1 — Document ingestion module

Accepts a local PDF, a local text file, or a Hugging Face dataset (domain-specific archive) and returns a single raw text string.

In [ ]:
def load_pdf(path):
    reader = PdfReader(path)
    text = ''
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + '\n'
    return text


def load_txt(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()


def load_hf_dataset(dataset_name, split='train', text_column='text', config=None, max_docs=200):
    from datasets import load_dataset
    ds = load_dataset(dataset_name, config, split=split) if config else load_dataset(dataset_name, split=split)
    texts = ds[text_column][:max_docs]
    return '\n'.join(texts)


def ingest(source_path=None, hf_dataset=None, **kwargs):
    """Single entry point for Step 1. Pass either source_path (pdf/txt) or hf_dataset (dataset name)."""
    if hf_dataset:
        return load_hf_dataset(hf_dataset, **kwargs)

    ext = source_path.lower().split('.')[-1]
    if ext == 'pdf':
        return load_pdf(source_path)
    elif ext in ('txt', 'md'):
        return load_txt(source_path)
    else:
        raise ValueError(f'Unsupported file type: {ext}')


Defines functions to read PDF or text files and extract their content.

### Upload and ingest a document (Colab)

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

raw_text = ingest(source_path=file_name)

print(f'Loaded {len(raw_text)} characters')
print(raw_text[:500])


Saving 2_Project Synopsis Format 2026-2027.pdf to 2_Project Synopsis Format 2026-2027.pdf
Loaded 5931 characters
Sanjivani Rural Education Society’s
Sanjivani College of Engineering, Kopargaon-423603
(An Autonomous Institute Affiliated to Savitribai Phule Pune University, Pune)
NAAC ‘A’ Grade Accredited
Department of Computer Engineering
UG Programme in Computer EngineeringTier1 NBA Accredited
FINAL YEAR PROJECT SYNOPSIS
(Academic Year 2026-2027)
Project Group Id 16
Title of the Project GestureConnect- Marathi Sign Language Communication Platform
Project Domain Artificial Intelligence (AI)
Problem Statemen


Uploads the document and loads its text into the notebook successfully.

## Step 2 — Clean chunking methodology

In [ ]:
def clean_text(text):
    text = re.sub(r'-\n', '', text)      # rejoin hyphenated line breaks
    text = re.sub(r'[ \t]+', ' ', text)   # collapse horizontal whitespace only
    text = re.sub(r'\n{2,}', '\n', text)  # collapse multiple blank lines
    return text.strip()


def split_into_units(text):
    """Line breaks are boundaries first (preserves table rows / form fields),
    long lines are further split into sentences."""
    units = []
    for line in text.split('\n'):
        line = line.strip()
        if not line:
            continue
        for sentence in re.split(r'(?<=[.!?])\s+', line):
            sentence = sentence.strip()
            if sentence:
                units.append(sentence)
    return units


def chunk_sentences(units, chunk_size=500, overlap_chars=100):
    chunks = []
    current = []
    current_len = 0

    for unit in units:
        if current and current_len + len(unit) + 1 > chunk_size:
            chunks.append(' '.join(current))

            # carry over overlap by character budget (not raw unit count) so an
            # oversized unit can't dominate multiple chunks in a row
            overlap, overlap_len = [], 0
            for u in reversed(current):
                if overlap_len + len(u) > overlap_chars:
                    break
                overlap.insert(0, u)
                overlap_len += len(u) + 1
            current, current_len = overlap, overlap_len

        current.append(unit)
        current_len += len(unit) + 1

    if current:
        chunks.append(' '.join(current))

    return chunks


def build_chunks(raw_text, chunk_size=500, overlap_chars=100):
    cleaned = clean_text(raw_text)
    units = split_into_units(cleaned)
    return chunk_sentences(units, chunk_size=chunk_size, overlap_chars=overlap_chars)


chunks = build_chunks(raw_text, chunk_size=500, overlap_chars=100)

print('Total chunks:', len(chunks))
print(chunks[0])


Total chunks: 15
Sanjivani Rural Education Society’s Sanjivani College of Engineering, Kopargaon-423603 (An Autonomous Institute Affiliated to Savitribai Phule Pune University, Pune) NAAC ‘A’ Grade Accredited Department of Computer Engineering UG Programme in Computer EngineeringTier1 NBA Accredited FINAL YEAR PROJECT SYNOPSIS (Academic Year 2026-2027) Project Group Id 16 Title of the Project GestureConnect- Marathi Sign Language Communication Platform Project Domain Artificial Intelligence (AI)


Cleans unnecessary characters from the text and divides it into smaller chunks.

## Step 3 — Map chunks to embeddings


In [ ]:
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


def embed_texts(texts):
    return embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)


chunk_embeddings = embed_texts(chunks)
print(chunk_embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(15, 384)


Converts text chunks into numerical vector embeddings using the Sentence Transformer model.

## Step 4 — Vector database

In [ ]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(np.array(chunk_embeddings).astype('float32'))
print('Vectors stored:', index.ntotal)

tokenized_chunks = [c.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)


Vectors stored: 15


Creates a FAISS index and stores all document embeddings for fast similarity search.

## Step 5 — Query embedding route


In [ ]:
def embed_query(question):
    return embed_model.encode([question], normalize_embeddings=True).astype('float32')


Converts the user's question into an embedding so it can be compared with stored document vectors.

## Step 6 — Retrieval module


In [ ]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


def vector_search(question, top_k=8):
    q_emb = embed_query(question)
    scores, indices = index.search(q_emb, top_k)
    return list(indices[0]), list(scores[0])


def bm25_search(question, top_k=8):
    tokenized_q = question.lower().split()
    scores = bm25.get_scores(tokenized_q)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return list(top_indices), [scores[i] for i in top_indices]


def hybrid_search(question, top_k=8, use_hybrid=True):
    vec_ids, _ = vector_search(question, top_k)
    if not use_hybrid:
        return vec_ids

    bm_ids, _ = bm25_search(question, top_k)

    rrf_scores = {}
    for rank, idx in enumerate(vec_ids):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (60 + rank)
    for rank, idx in enumerate(bm_ids):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (60 + rank)

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in fused[:top_k]]


def retrieve_context(question, top_k=8, final_k=4, use_hybrid=True, use_rerank=True):
    candidate_ids = hybrid_search(question, top_k=top_k, use_hybrid=use_hybrid)
    candidates = [chunks[i] for i in candidate_ids]

    if use_rerank and candidates:
        pairs = [[question, c] for c in candidates]
        scores = reranker.predict(pairs)
        candidates = [c for c, _ in sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)]

    top_chunks = candidates[:final_k]
    return '\n\n'.join(top_chunks), top_chunks


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retrieves the most relevant document chunks using vector search, BM25, and reranking techniques.

## Step 7 — Introduces answer generation using an LLM



In [ ]:
MODEL_NAME = 'google/flan-t5-base'  # try 'google/flan-t5-large' for better accuracy if resources allow

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f'{MODEL_NAME} loaded successfully')


def answer_question(question, top_k=8, final_k=4, use_hybrid=True, use_rerank=True):
    context, _ = retrieve_context(
        question, top_k=top_k, final_k=final_k, use_hybrid=use_hybrid, use_rerank=use_rerank
    )

    prompt = f"""Answer the question using ONLY the context below. Be concise and extract the exact answer.
If the answer is not present in the context, reply exactly: The document does not contain that information.

Context:
{context}

Question: {question}

Answer:"""

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer, context


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

google/flan-t5-base loaded successfully


Loads the FLAN-T5 model and generates answers based on the retrieved document context.

## Test with document

In [ ]:
question = 'What are the names of the students in the project group?'

answer, context = answer_question(question)

print('Question:\n', question)
print('\nRetrieved Context:\n')
print(context)
print('\nAnswer:\n')
print(answer)


Question:
 What are the names of the students in the project group?

Retrieved Context:

Main Outcome: A tested, evaluated, documented, and final-ready GestureConnect system. Name of Students in the Project Group with their 1. Pratap Satish Navale 2. Maithili Anil Pawar signatures 3. Pranjali Shriram Patil 4. Hardik Bhagwan Sonawane Name of the Guide and Signature Prof. Swapnali Gawali Date Project title Accepted/Rejected. Prof. Swapnali Gawali Dr. M. A. Jawale Project Guide HOD

Sanjivani Rural Education Society’s Sanjivani College of Engineering, Kopargaon-423603 (An Autonomous Institute Affiliated to Savitribai Phule Pune University, Pune) NAAC ‘A’ Grade Accredited Department of Computer Engineering UG Programme in Computer EngineeringTier1 NBA Accredited FINAL YEAR PROJECT SYNOPSIS (Academic Year 2026-2027) Project Group Id 16 Title of the Project GestureConnect- Marathi Sign Language Communication Platform Project Domain Artificial Intelligence (AI)

Project Domain Artificial Inte

Tested the system by asking one question and displays the generated answer successfully.

## More test questions

In [ ]:
questions = [
    'What is the title of the project?',
    'What is the project domain?',
    'What is the project group ID?',
    'Which department is associated with this project?',
    'Which college is this project from?',
    'What is the academic year of the project synopsis?',
    'Who is the project guide?',
    'Who is the Head of Department mentioned in the document?',
    'How many students are in the project group?',
    'What are the names of the students in the project group?'
]

for q in questions:
    answer, _ = answer_question(q)
    print('=' * 70)
    print('Question:', q)
    print('Answer:', answer)


Question: What is the title of the project?
Answer: GestureConnect- Marathi Sign Language Communication Platform
Question: What is the project domain?
Answer: Marathi Sign Language
Question: What is the project group ID?
Answer: 16
Question: Which department is associated with this project?
Answer: Computer Engineering
Question: Which college is this project from?
Answer: Sanjivani College of Engineering
Question: What is the academic year of the project synopsis?
Answer: 2026-2027
Question: Who is the project guide?
Answer: Prof. Swapnali Gawali
Question: Who is the Head of Department mentioned in the document?
Answer: Prof. Swapnali Gawali
Question: How many students are in the project group?
Answer: 16
Question: What are the names of the students in the project group?
Answer: Pratap Satish Navale 2. Maithili Anil Pawar signatures 3. Pranjali Shriram Patil 4. Hardik Bhagwan Sonawane


Tested the system by asking one question and displays the generated answer.

## Step 8 — Optimization experiments


In [ ]:
configs = [
    {'name': 'vector only', 'use_hybrid': False, 'use_rerank': False},
    {'name': 'hybrid (vector + BM25)', 'use_hybrid': True, 'use_rerank': False},
    {'name': 'hybrid + rerank', 'use_hybrid': True, 'use_rerank': True},
]

test_question = 'What is the title of the project?'

for cfg in configs:
    ans, _ = answer_question(test_question, use_hybrid=cfg['use_hybrid'], use_rerank=cfg['use_rerank'])
    print(f"[{cfg['name']}] -> {ans}")


[vector only] -> GestureConnect
[hybrid (vector + BM25)] -> GestureConnect- Marathi Sign Language Communication Platform Project
[hybrid + rerank] -> GestureConnect- Marathi Sign Language Communication Platform


Compares different retrieval methods such as vector search, hybrid search, and reranking to improve accuracy.

### Chunk-boundary experiment


In [ ]:
chunk_configs = [
    {'chunk_size': 300, 'overlap_chars': 50},
    {'chunk_size': 500, 'overlap_chars': 100},
    {'chunk_size': 800, 'overlap_chars': 150},
]

for cfg in chunk_configs:
    trial_chunks = build_chunks(raw_text, chunk_size=cfg['chunk_size'], overlap_chars=cfg['overlap_chars'])
    print(f"chunk_size={cfg['chunk_size']}, overlap_chars={cfg['overlap_chars']} -> {len(trial_chunks)} chunks")


chunk_size=300, overlap_chars=50 -> 24 chunks
chunk_size=500, overlap_chars=100 -> 15 chunks
chunk_size=800, overlap_chars=150 -> 9 chunks


Tested different chunk sizes and overlaps to determine which configuration gives better retrieval results.

In [ ]:
# Step 8c — Verification tests: confirm retrieval + optimizations produce correct answers

# Expected substrings for your GestureConnect synopsis PDF — adjust if you test a different document
expected_answers = {
    'What is the title of the project?': 'GestureConnect',
    'What is the project domain?': 'Artificial Intelligence',
    'What is the project group ID?': '16',
    'Which college is this project from?': 'Sanjivani',
    'Who is the project guide?': 'Swapnali Gawali',
    'Who is the Head of Department mentioned in the document?': 'Jawale',
    'How many students are in the project group?': '4',
}

def check_answer(answer, expected_substring):
    return expected_substring.lower() in answer.lower()


def run_test_suite(use_hybrid, use_rerank, label):
    print('\n')
    print(f'CONFIG: {label}  (hybrid={use_hybrid}, rerank={use_rerank})')


    passed_count = 0
    for question, expected in expected_answers.items():
        answer, _ = answer_question(question, use_hybrid=use_hybrid, use_rerank=use_rerank)
        passed = check_answer(answer, expected)
        passed_count += passed

        print('-' * 70)
        print('Q:', question)
        print('Expected to contain:', expected)
        print('Got:', answer)
        print('Result:', 'PASS' if passed else 'FAIL')

    print('-' * 70)
    print(f'SCORE: {passed_count}/{len(expected_answers)} passed\n')
    return passed_count


# Run the same test suite across configs so the optimization's effect is measurable, not just claimed
score_baseline = run_test_suite(use_hybrid=False, use_rerank=False, label='vector only')
score_hybrid = run_test_suite(use_hybrid=True, use_rerank=False, label='hybrid (vector + BM25)')
score_hybrid_rerank = run_test_suite(use_hybrid=True, use_rerank=True, label='hybrid + rerank')

print('=' * 70)
print('SUMMARY')
print('=' * 70)
print(f'vector only        : {score_baseline}/{len(expected_answers)}')
print(f'hybrid              : {score_hybrid}/{len(expected_answers)}')
print(f'hybrid + rerank      : {score_hybrid_rerank}/{len(expected_answers)}')


######################################################################
CONFIG: vector only  (hybrid=False, rerank=False)
######################################################################
----------------------------------------------------------------------
Q: What is the title of the project?
Expected to contain: GestureConnect
Got: GestureConnect
Result: PASS
----------------------------------------------------------------------
Q: What is the project domain?
Expected to contain: Artificial Intelligence
Got: Artificial Intelligence
Result: PASS
----------------------------------------------------------------------
Q: What is the project group ID?
Expected to contain: 16
Got: 16
Result: PASS
----------------------------------------------------------------------
Q: Which college is this project from?
Expected to contain: Sanjivani
Got: Sanjivani College of Engineering
Result: PASS
----------------------------------------------------------------------
Q: Who is the project guide?


The Vector Only retrieval method achieved the highest accuracy (5/7). The Hybrid and Hybrid + Rerank methods both achieved 4/7 accuracy, indicating that adding keyword search and reranking did not improve the results for the given set of test questions.